# Árvores de decisão

**Objetivo:** treinar uma árvore no Iris, ler as perguntas que ela aprendeu, ver a profundidade controlar o overfitting (a curva treino × validação) e desenhar a fronteira retangular.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, cross_val_score

iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=SEMENTE, stratify=y)
print("treino:", X_tr.shape[0], "| teste:", X_te.shape[0])

## 1. As perguntas que a árvore aprendeu

Uma árvore rasa (profundidade 3) já classifica bem o Iris. O `export_text` imprime a árvore como um fluxograma de perguntas — leitura direta, sem caixa-preta.

In [ ]:
arvore = DecisionTreeClassifier(max_depth=3, random_state=SEMENTE)
arvore.fit(X_tr, y_tr)
print("acuracia no teste:", round(arvore.score(X_te, y_te), 3))
print()
print(export_text(arvore, feature_names=list(iris.feature_names)))

## 2. Profundidade × overfitting

Para cada profundidade, comparamos a acurácia no **treino** com a de **validação cruzada**. O treino sobe sempre em direção a 100%; a validação faz um U — sinal de que árvores fundas memorizam.

In [ ]:
profundidades = list(range(1, 12))
acc_treino = []
acc_validacao = []
for d in profundidades:
    modelo = DecisionTreeClassifier(max_depth=d, random_state=SEMENTE)
    modelo.fit(X_tr, y_tr)
    acc_treino.append(modelo.score(X_tr, y_tr))
    acc_validacao.append(cross_val_score(modelo, X_tr, y_tr, cv=5).mean())
    print("prof", str(d).rjust(2), "| treino", round(acc_treino[-1], 3),
          "| validacao", round(acc_validacao[-1], 3))

In [ ]:
figura = go.Figure()
figura.add_trace(go.Scatter(x=profundidades, y=acc_treino, mode="lines+markers",
                            line=dict(color=AZUL), name="treino"))
figura.add_trace(go.Scatter(x=profundidades, y=acc_validacao, mode="lines+markers",
                            line=dict(color=VERMELHO), name="validacao (CV)"))
figura.update_layout(title="Arvore: profundidade x acuracia",
                     xaxis_title="max_depth", yaxis_title="acuracia", height=360,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 3. A fronteira retangular

Com dois preditores, a fronteira da árvore é feita de **retângulos** (cortes paralelos aos eixos) — a assinatura visual do modelo.

In [ ]:
X2 = X[:, 2:4]
arvore2 = DecisionTreeClassifier(max_depth=4, random_state=SEMENTE).fit(X2, y)
passo = 0.02
gx, gy = np.meshgrid(np.arange(X2[:, 0].min()-0.5, X2[:, 0].max()+0.5, passo),
                     np.arange(X2[:, 1].min()-0.5, X2[:, 1].max()+0.5, passo))
zz = arvore2.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

figura = go.Figure()
figura.add_trace(go.Heatmap(x=gx[0], y=gy[:, 0], z=zz, showscale=False,
                            colorscale="Blugrn", opacity=0.35))
figura.add_trace(go.Scatter(x=X2[:, 0], y=X2[:, 1], mode="markers",
                            marker=dict(color=y, colorscale="Blugrn", size=6,
                                        line=dict(width=0.5, color="white"))))
figura.update_layout(title="Fronteira retangular da arvore (2 preditores)",
                     height=400, margin=dict(l=10, r=10, t=50, b=10), showlegend=False)
figura.show()

## Exercício

Pela curva do item 2, qual `max_depth` você escolheria para este problema? Justifique com base na acurácia de validação, não na de treino.

<details><summary>Ver resposta</summary>

Escolhe-se a menor profundidade que já atinge o platô da acurácia de **validação** — no Iris, tipicamente `max_depth` entre 3 e 4. Ir além disso só aumenta a acurácia de treino (rumo a 100%) sem ganho na validação, o que é overfitting: mais complexidade sem mais generalização. A regra é preferir o modelo mais simples que empata no topo da validação.

</details>